In [ ]:
# =============================================================================
# DATA CLEANING PIPELINE for RavenStack 5 CSV Files
# =============================================================================

import pandas as pd
import numpy as np
from datetime import datetime
import re
import os

# -----------------------------------------------------------------------------
# 1. GENERAL CLEANING UTILITIES
# -----------------------------------------------------------------------------

def clean_string_columns(df, columns):
    """Striping leading/trailing whitespace and replace empty strings with NaN."""
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace(['nan', 'None', ''], np.nan)
    return df

def convert_boolean_columns(df, columns):
    """Converting string booleans (TRUE/FALSE, true/false, 1/0) to bool."""
    bool_map = {
        'TRUE': True, 'FALSE': False,
        'true': True, 'false': False,
        'True': True, 'False': False,
        '1': True, '0': False,
        'Yes': True, 'No': False,
        'yes': True, 'no': False
    }
    for col in columns:
        if col in df.columns:
            df[col] = df[col].map(bool_map).astype('boolean')  
    return df

def convert_numeric_columns(df, columns, coerce=True):
    """Convert columns to numeric, coercing errors to NaN."""
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def convert_date_columns(df, columns, format='%d-%m-%Y'):
    """Convert date columns (without time) to datetime."""
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format=format, errors='coerce')
    return df

def convert_datetime_columns(df, columns, format='%d-%m-%Y %H:%M'):
    """Convert datetime columns (with time) to datetime."""
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format=format, errors='coerce')
    return df

def fill_missing_numeric(df, columns, strategy='median'):
    """Fill missing numeric values with median/mean/mode."""
    for col in columns:
        if col in df.columns and df[col].dtype in ['float64', 'int64', 'Float64', 'Int64']:
            if strategy == 'median':
                df[col].fillna(df[col].median(), inplace=True)
            elif strategy == 'mean':
                df[col].fillna(df[col].mean(), inplace=True)
            elif strategy == 'mode':
                df[col].fillna(df[col].mode()[0], inplace=True)
            elif strategy == 'zero':
                df[col].fillna(0, inplace=True)
    return df

def fill_missing_categorical(df, columns, fill_value='Unknown'):
    """Fill missing categorical/string columns."""
    for col in columns:
        if col in df.columns and df[col].dtype == 'object':
            df[col].fillna(fill_value, inplace=True)
    return df

def drop_duplicates(df, subset=None):
    """Drop duplicate rows."""
    if subset:
        df = df.drop_duplicates(subset=subset, keep='first')
    else:
        df = df.drop_duplicates(keep='first')
    return df

def validate_foreign_keys(df, ref_df, key_col, ref_key_col, table_name='', ref_table=''):
    """Check if all keys in df exist in ref_df. Prints warnings."""
    if key_col not in df.columns or ref_key_col not in ref_df.columns:
        return
    missing = df[~df[key_col].isin(ref_df[ref_key_col])]
    if not missing.empty:
        print(f"⚠️  {len(missing)} rows in {table_name} have {key_col} not found in {ref_table}.{ref_key_col}")

# -----------------------------------------------------------------------------
# 2. TABLE-SPECIFIC CLEANING FUNCTIONS
# -----------------------------------------------------------------------------

def clean_accounts(df):
    """Clean ravenstack_accounts.csv"""
    print("Cleaning accounts...")
    # Strip whitespace from all string columns
    str_cols = ['account_id', 'account_name', 'industry', 'country', 'referral_source', 'plan_tier']
    df = clean_string_columns(df, str_cols)
    
    # Convert signup_date
    df = convert_date_columns(df, ['signup_date'], format='%d-%m-%Y')
    
    # Convert booleans
    bool_cols = ['is_trial', 'churn_flag']
    df = convert_boolean_columns(df, bool_cols)
    
    # Convert seats to numeric (integer)
    df = convert_numeric_columns(df, ['seats'])
    # Fill missing seats with median 
    df = fill_missing_numeric(df, ['seats'], strategy='median')
    # Ensure integer type
    if 'seats' in df.columns:
        df['seats'] = df['seats'].astype('Int64')
    
    # Fill missing categorical
    cat_cols = ['industry', 'country', 'referral_source', 'plan_tier']
    df = fill_missing_categorical(df, cat_cols, fill_value='Unknown')
    
    # Drop duplicates on account_id
    df = drop_duplicates(df, subset=['account_id'])
    
    return df

def clean_churn_events(df):
    """Clean ravenstack_churn_events.csv"""
    print("Cleaning churn_events...")
    str_cols = ['churn_event_id', 'account_id', 'reason_code', 'feedback_text']
    df = clean_string_columns(df, str_cols)
    
    # Convert churn_date
    df = convert_date_columns(df, ['churn_date'], format='%d-%m-%Y')
    
    # Booleans
    bool_cols = ['preceding_upgrade_flag', 'preceding_downgrade_flag', 'is_reactivation']
    df = convert_boolean_columns(df, bool_cols)
    
    # Numeric refund_amount
    df = convert_numeric_columns(df, ['refund_amount_usd'])
    # Fill missing refund amount with 0 (also filling for no refunds)
    df = fill_missing_numeric(df, ['refund_amount_usd'], strategy='zero')
    
    # Fill missing feedback_text with 'No feedback'
    df = fill_missing_categorical(df, ['feedback_text'], fill_value='No feedback')
    
    # Drop duplicates
    df = drop_duplicates(df, subset=['churn_event_id'])
    
    return df

def clean_feature_usage(df):
    """Clean ravenstack_feature_usage.csv"""
    print("Cleaning feature_usage...")
    str_cols = ['usage_id', 'subscription_id', 'feature_name']
    df = clean_string_columns(df, str_cols)
    
    # usage_date
    df = convert_date_columns(df, ['usage_date'], format='%d-%m-%Y')
    
    # Booleans
    df = convert_boolean_columns(df, ['is_beta_feature'])
    
    # Numeric counts/durations
    num_cols = ['usage_count', 'usage_duration_secs', 'error_count']
    df = convert_numeric_columns(df, num_cols)
    # Fill missing numeric with 0 
    df = fill_missing_numeric(df, num_cols, strategy='zero')
    # Ensure integer types
    for col in ['usage_count', 'usage_duration_secs', 'error_count']:
        if col in df.columns:
            df[col] = df[col].astype('Int64')
    
    # Drop duplicates
    df = drop_duplicates(df, subset=['usage_id'])
    
    return df

def clean_subscriptions(df):
    """Clean ravenstack_subscriptions.csv"""
    print("Cleaning subscriptions...")
    str_cols = ['subscription_id', 'account_id', 'plan_tier', 'billing_frequency']
    df = clean_string_columns(df, str_cols)
    
    # Dates: start_date and end_date
    df = convert_date_columns(df, ['start_date'], format='%d-%m-%Y')
    df = convert_date_columns(df, ['end_date'], format='%d-%m-%Y')
    
    # Booleans
    bool_cols = ['is_trial', 'upgrade_flag', 'downgrade_flag', 'churn_flag', 'auto_renew_flag']
    df = convert_boolean_columns(df, bool_cols)
    
    # Numeric: seats, mrr_amount, arr_amount
    num_cols = ['seats', 'mrr_amount', 'arr_amount']
    df = convert_numeric_columns(df, num_cols)
    # Fill missing seats with median, monetary with 0 
    df = fill_missing_numeric(df, ['seats'], strategy='median')
    df = fill_missing_numeric(df, ['mrr_amount', 'arr_amount'], strategy='zero')
    # Ensure integer types for seats, but keep float for monetary
    if 'seats' in df.columns:
        df['seats'] = df['seats'].astype('Int64')
    
    # Fill missing categorical (billing_frequency, plan_tier) with mode
    cat_cols = ['plan_tier', 'billing_frequency']
    df = fill_missing_categorical(df, cat_cols, fill_value='Unknown')
    
    # Drop duplicates
    df = drop_duplicates(df, subset=['subscription_id'])
    
    return df

def clean_support_tickets(df):
    """Clean ravenstack_support_tickets.csv"""
    print("Cleaning support_tickets...")
    str_cols = ['ticket_id', 'account_id', 'priority']
    df = clean_string_columns(df, str_cols)
    
    # submitted_at: only date
    df = convert_date_columns(df, ['submitted_at'], format='%d-%m-%Y')
    # closed_at: date + time
    df = convert_datetime_columns(df, ['closed_at'], format='%d-%m-%Y %H:%M')
    
    # Numeric: resolution_time_hours, first_response_time_minutes, satisfaction_score
    num_cols = ['resolution_time_hours', 'first_response_time_minutes', 'satisfaction_score']
    df = convert_numeric_columns(df, num_cols)
    # Fill missing satisfaction_score with median
    df = fill_missing_numeric(df, ['satisfaction_score'], strategy='median')
    # Fill missing resolution_time or first_response_time with 0 
    df = fill_missing_numeric(df, ['resolution_time_hours', 'first_response_time_minutes'], strategy='zero')
    # Ensure integer types for these numeric columns
    for col in ['resolution_time_hours', 'first_response_time_minutes']:
        if col in df.columns:
            df[col] = df[col].astype('Int64')
   
    
    # Boolean: escalation_flag
    df = convert_boolean_columns(df, ['escalation_flag'])
    
    # Fill missing priority with 'unknown'
    df = fill_missing_categorical(df, ['priority'], fill_value='unknown')
    
    # Drop duplicates
    df = drop_duplicates(df, subset=['ticket_id'])
    
    return df

# -----------------------------------------------------------------------------
# 3. MAIN PIPELINE EXECUTION
# -----------------------------------------------------------------------------

def run_pipeline(file_list=None):
    
    if file_list is None:
        file_list = [
            'ravenstack_accounts.csv',
            'ravenstack_churn_events.csv',
            'ravenstack_feature_usage.csv',
            'ravenstack_subscriptions.csv',
            'ravenstack_support_tickets.csv'
        ]
    
    # Dictionary to hold cleaned dataframes for foreign key validation
    cleaned_dfs = {}
    
    # Cleaning function mapping
    clean_map = {
        'ravenstack_accounts.csv': clean_accounts,
        'ravenstack_churn_events.csv': clean_churn_events,
        'ravenstack_feature_usage.csv': clean_feature_usage,
        'ravenstack_subscriptions.csv': clean_subscriptions,
        'ravenstack_support_tickets.csv': clean_support_tickets
    }
    
    for file in file_list:
        if not os.path.exists(file):
            print(f"⚠️  File not found: {file} - skipping")
            continue
        
        print(f"\n📂 Loading {file}...")
        df = pd.read_csv(file, encoding='utf-8', keep_default_na=True, na_values=[''])
        print(f"   Shape: {df.shape}")
        
        # Clean using appropriate function
        clean_func = clean_map.get(file)
        if clean_func:
            df_clean = clean_func(df)
        else:
            print(f"⚠️  No cleaning function defined for {file} - using raw")
            df_clean = df
        
        # Save cleaned version
        base, ext = os.path.splitext(file)
        out_file = f"{base}_cleaned{ext}"
        df_clean.to_csv(out_file, index=False, encoding='utf-8')
        print(f"✅ Saved cleaned file: {out_file} (shape: {df_clean.shape})")
        
        
        cleaned_dfs[base] = df_clean
    
    # -------------------------------------------------------------------------
    # 4. FOREIGN KEY VALIDATION (optional, but informative)
    # -------------------------------------------------------------------------
    print("\n" + "="*60)
    print("FOREIGN KEY VALIDATION")
    print("="*60)
    
    # Accounts -> account_id
    if 'ravenstack_accounts' in cleaned_dfs:
        accounts = cleaned_dfs['ravenstack_accounts']
        # Check subscriptions.account_id
        if 'ravenstack_subscriptions' in cleaned_dfs:
            validate_foreign_keys(cleaned_dfs['ravenstack_subscriptions'], accounts,
                                  'account_id', 'account_id', 'subscriptions', 'accounts')
        # Check churn_events.account_id
        if 'ravenstack_churn_events' in cleaned_dfs:
            validate_foreign_keys(cleaned_dfs['ravenstack_churn_events'], accounts,
                                  'account_id', 'account_id', 'churn_events', 'accounts')
        # Check support_tickets.account_id
        if 'ravenstack_support_tickets' in cleaned_dfs:
            validate_foreign_keys(cleaned_dfs['ravenstack_support_tickets'], accounts,
                                  'account_id', 'account_id', 'support_tickets', 'accounts')
    
    # Subscriptions -> subscription_id for feature_usage
    if 'ravenstack_subscriptions' in cleaned_dfs and 'ravenstack_feature_usage' in cleaned_dfs:
        subs = cleaned_dfs['ravenstack_subscriptions']
        validate_foreign_keys(cleaned_dfs['ravenstack_feature_usage'], subs,
                              'subscription_id', 'subscription_id', 'feature_usage', 'subscriptions')
    
    print("\n🎉 Data cleaning pipeline completed successfully!")
    print("   Cleaned CSV files are ready for analysis.")

# -----------------------------------------------------------------------------
# 5. EXECUTE THE PIPELINE
# -----------------------------------------------------------------------------

if __name__ == "__main__":
    run_pipeline()


📂 Loading ravenstack_accounts.csv...
   Shape: (500, 10)
Cleaning accounts...
✅ Saved cleaned file: ravenstack_accounts_cleaned.csv (shape: (500, 10))

📂 Loading ravenstack_churn_events.csv...
   Shape: (600, 9)
Cleaning churn_events...
✅ Saved cleaned file: ravenstack_churn_events_cleaned.csv (shape: (600, 9))

📂 Loading ravenstack_feature_usage.csv...


C:\Users\WINDOWS\AppData\Local\Temp\ipykernel_3056\1588689136.py:64: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\WINDOWS\AppData\Local\Temp\ipykernel_3056\1588689136.py:77: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

   Shape: (25000, 8)
Cleaning feature_usage...


C:\Users\WINDOWS\AppData\Local\Temp\ipykernel_3056\1588689136.py:70: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(0, inplace=True)


✅ Saved cleaned file: ravenstack_feature_usage_cleaned.csv (shape: (24979, 8))

📂 Loading ravenstack_subscriptions.csv...
   Shape: (5000, 14)
Cleaning subscriptions...


C:\Users\WINDOWS\AppData\Local\Temp\ipykernel_3056\1588689136.py:64: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\WINDOWS\AppData\Local\Temp\ipykernel_3056\1588689136.py:70: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

✅ Saved cleaned file: ravenstack_subscriptions_cleaned.csv (shape: (5000, 14))

📂 Loading ravenstack_support_tickets.csv...
   Shape: (2000, 9)
Cleaning support_tickets...
✅ Saved cleaned file: ravenstack_support_tickets_cleaned.csv (shape: (2000, 9))

FOREIGN KEY VALIDATION

🎉 Data cleaning pipeline completed successfully!
   Cleaned CSV files are ready for analysis.
